In [13]:
import pandas as pd
import numpy as np
import pysrt
import os
import subprocess
from pathlib import Path
import json

In [3]:
# define if we run or load the matches dataframe
load_matches_df = False

In [4]:
# read signbank csv
file_path = r"..\..\..\AnimData\dictionary-export.csv"
df = pd.read_csv(file_path)

df.head()

,Signbank ID,Dataset,Lemma ID Gloss (Dutch),Lemma ID Gloss (English),Annotation ID Gloss (Dutch),Annotation ID Gloss (English),Senses (Dutch),Senses (English),Example Sentences (Dutch),Example Sentences (English),...,In the Web dictionary,Is this a proposed new sign?,Dialects,Sequential Morphology,Simultaneous Morphology,Blend Morphology,Relations to other signs,Relations to foreign signs,Tags,Notes
0,3808,NGT,#A,NaN,#A,#A,1. a...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3809,NGT,#B,NaN,#B,#B,1. b...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3810,NGT,#C,NaN,#C,#C,1. c...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3811,NGT,#D,NaN,#D,#D,1. d...,1. d...,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3812,NGT,#E,NaN,#E,#E,1. e...,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def get_handshape_from_gloss(gloss):
    row = df[df['Annotation ID Gloss (Dutch)'] == gloss]
    if not row.empty:
        return row.iloc[0]['Strong Hand']
    return None

In [6]:
def get_frames_from_timecode(timecode, fps):
    # Convert timecode to total seconds
    total_seconds = timecode.hours * 3600 + timecode.minutes * 60 + timecode.seconds + timecode.milliseconds / 1000.0
    # Calculate frame number
    frame_number = int(total_seconds * fps)
    return frame_number

In [7]:
# read srt
test = pysrt.open(r"..\..\..\AnimData\srt\M20240925_1858_260319_0_Signbank_ID_glossen.srt")

fps = 30  # frames per second

for sub in test:
    print(get_frames_from_timecode(sub.start, fps), get_frames_from_timecode(sub.end, fps), get_handshape_from_gloss(sub.text))

10 26 V
33 39 S
49 62 1
71 81 T
90 128 -


In [8]:
def get_matches_from_csv(csv_path):
    matches = []
    with open(csv_path, 'r') as f:
        next(f)  # Skip the header line
        for line in f:
            # rank, handshape_file, distance
            parts = line.strip().split(',')
            if len(parts) == 3:
                matches.append((int(parts[0]), parts[1], float(parts[2])))
    return matches

In [9]:
if not load_matches_df:

    # go through each fbx and compare
    exe_loc = r"..\..\build\Release\shapeseg.exe"
    repo_root = Path(exe_loc).resolve().parents[2]
    exe = repo_root / "build" / "Release" / "shapeseg.exe"

    fbx_folder = repo_root / "fbx" / "palmer"

    segment_file = r"..\..\output\handshape_segments.csv"

    matches = dict()
    segments_by_fbx = dict()

    for fbx_file in os.listdir(fbx_folder):
        if fbx_file.endswith(".fbx"):
            matches[fbx_file] = []

            fbx = fbx_folder / fbx_file

            # Update output files by running the command
            result = subprocess.run(
                [str(exe), "s", str(fbx)],
                cwd=repo_root,
                check=True,
                capture_output=True,
                text=True,
            )

            # Get number of segments
            file = open(segment_file, 'r')
            segments = [(line.split(',')[0], line.split(',')[1]) for line in file.readlines()[1:]]
            file.close()
            segments_by_fbx[fbx_file] = segments
            print(f"Processed {fbx_file}: {len(segments)} segments found.")

            # Get the matches for each segment
            for i in range(len(segments)):
                result = subprocess.run(
                    [str(exe), "m", str(fbx), str(i)],
                    cwd=repo_root,
                    check=True,
                    capture_output=True,
                    text=True,
                )

                # get matches
                match_file = repo_root / "output" / "handshape_matches.csv"
                match_list = get_matches_from_csv(match_file)
                matches[fbx_file].append(match_list)



Processed M20240925_1858_260319_0.fbx: 10 segments found.
Processed M20240925_1872_260318_0.fbx: 9 segments found.
Processed M20240925_2050_260319_1.fbx: 5 segments found.
Processed M20241106_5930_260319_2.fbx: 7 segments found.
Processed M20241106_5991_260318_5.fbx: 7 segments found.
Processed M20241106_6008_260319_1.fbx: 7 segments found.
Processed M20241106_6023_260318_2.fbx: 6 segments found.
Processed M20241106_6040_260318_0.fbx: 7 segments found.
Processed M20241106_6044_260318_1.fbx: 9 segments found.
Processed M20241106_6134_260319_1.fbx: 6 segments found.
Processed M20241107_6192_260317_0.fbx: 8 segments found.
Processed M20241107_6205_260318_0.fbx: 4 segments found.
Processed M20241107_6206_260318_0.fbx: 6 segments found.
Processed M20241107_6240_260318_0.fbx: 6 segments found.
Processed M20241107_6338_260318_0.fbx: 7 segments found.
Processed M20241107_6355_260318_0.fbx: 8 segments found.
Processed M20241107_6361_260318_1.fbx: 7 segments found.
Processed M20241107_6363_26031

In [10]:
if not load_matches_df:

    # make a dataframe from the matches dictionary
    data = []

    for fbx_file, match_lists in matches.items():
        for segment_idx, match_list in enumerate(match_lists):
            for match in match_list:
                data.append({
                    "fbx_file": fbx_file,
                    "segment": segment_idx,
                "rank": match[0],
                "handshape_file": match[1],
                "distance": match[2]
            })
            segment_idx += 1

    matches_df = pd.DataFrame(data)

    matches_df.head()

In [11]:
if not load_matches_df:

    # Convert handshape_file to handshape
    matches_df['handshape'] = (
        matches_df['handshape_file']
        .str.rsplit('/', n=1).str[-1]
        .str.split(r'[-.]', regex=True).str[0]
    )

    matches_df.head()

In [14]:
if not load_matches_df:

    # save the matches dataframe to a csv file
    output_csv_path = repo_root / "output" / "handshape_matches_dataframe.csv"
    matches_df.to_csv(output_csv_path, index=False)

    # save segments_by_fbx dict to a json file
    output_json_path = repo_root / "output" / "handshape_segments_by_fbx.json"
    json.dump(segments_by_fbx, open(output_json_path, 'w'), indent=4)


In [16]:
if load_matches_df:
    # load the matches dataframe from a csv file
    output_csv_path = repo_root / "output" / "handshape_matches_dataframe.csv"
    matches_df = pd.read_csv(output_csv_path)

    # load segments_by_fbx from a json file
    output_json_path = repo_root / "output" / "handshape_segments_by_fbx.json"
    segments_by_fbx = json.load(open(output_json_path, 'r'))

In [17]:
srt_folder = r"..\..\..\AnimData\srt"

def load_srt(fbx_file):
    srt_path = Path(srt_folder) / f"{Path(fbx_file).stem}_Signbank_ID_glossen.srt"
    if not srt_path.exists():
        print(f"No srt found for {fbx_file}: {srt_path}")
        return None
    return pysrt.open(str(srt_path))

In [18]:
def find_overlapping_subs_from_segment(fbx_file, segment_idx):
    srt = load_srt(fbx_file)
    if srt is None:
        return []

    # Get the start and end frames of the segment
    segment_start_frame = int(segments_by_fbx[fbx_file][segment_idx][0])
    segment_end_frame = int(segments_by_fbx[fbx_file][segment_idx][1])

    overlapping_subs = []
    for sub in srt:
        sub_start_frame = get_frames_from_timecode(sub.start, fps)
        sub_end_frame = get_frames_from_timecode(sub.end, fps)

        # Check for overlap
        if not (sub_end_frame < segment_start_frame or sub_start_frame > segment_end_frame):
            overlapping_subs.append(sub)

    if not overlapping_subs:
        print(f"No overlapping subtitles found for {fbx_file}, segment {segment_idx} (frames {segment_start_frame}-{segment_end_frame})")

    return overlapping_subs

In [19]:
# for each unique fbx_file, segment combination, find the rank of the correct handshape
def get_correct_handshape(fbx_file, segment_idx):
    # Get overlapping subs
    overlapping_subs = find_overlapping_subs_from_segment(fbx_file, segment_idx)

    if not overlapping_subs:
        return []

    # Get the correct handshape from the overlapping subs
    contender_handshapes = []
    for sub in overlapping_subs:
        print(f"Found overlapping subtitle for {fbx_file}, segment {segment_idx}: {sub.text}")
        contender_handshapes.append(get_handshape_from_gloss(sub.text))

    # Remove None values
    contender_handshapes = [hs for hs in contender_handshapes if hs is not None]

    if not contender_handshapes:
        print(f"No valid handshapes found for {fbx_file}, segment {segment_idx}")
        return []

    # Get the matches for this fbx_file and segment_idx
    matches_for_segment = matches_df[(matches_df['fbx_file'] == fbx_file) & (matches_df['segment'] == segment_idx)]

    if matches_for_segment.empty:
        print(f"No matches found for {fbx_file}, segment {segment_idx}")
        return []

    # Find the rank of the correct handshape
    correct_ranks = []
    for handshape in contender_handshapes:
        print(f"Looking for handshape '{handshape}' in matches for {fbx_file}, segment {segment_idx}")
        match_row = matches_for_segment[matches_for_segment['handshape'] == handshape]
        if not match_row.empty:
            correct_ranks.append(match_row.iloc[0]['rank'])
        else:
            print(f"Handshape '{handshape}' not found in matches for {fbx_file}, segment {segment_idx}")

    return correct_ranks

In [20]:
# Get correct ranks for all unique fbx_file, segment combinations
ranks = dict()

for fbx_file in matches_df['fbx_file'].unique():
    ranks[fbx_file] = dict()
    for segment_idx in matches_df[matches_df['fbx_file'] == fbx_file]['segment'].unique():
        correct_handshapes = get_correct_handshape(fbx_file, segment_idx)
        print(f"Correct handshapes for {fbx_file}, segment {segment_idx}: {correct_handshapes}")
        if correct_handshapes:
            ranks[fbx_file][segment_idx] = correct_handshapes


print(ranks)

Found overlapping subtitle for M20240925_1858_260319_0.fbx, segment 0: TWEEEN-A
Looking for handshape 'V' in matches for M20240925_1858_260319_0.fbx, segment 0
Correct handshapes for M20240925_1858_260319_0.fbx, segment 0: [np.int64(6)]
Found overlapping subtitle for M20240925_1858_260319_0.fbx, segment 1: KUNNEN-E
Looking for handshape 'S' in matches for M20240925_1858_260319_0.fbx, segment 1
Correct handshapes for M20240925_1858_260319_0.fbx, segment 1: [np.int64(5)]
Found overlapping subtitle for M20240925_1858_260319_0.fbx, segment 2: PT-1hand
Looking for handshape '1' in matches for M20240925_1858_260319_0.fbx, segment 2
Correct handshapes for M20240925_1858_260319_0.fbx, segment 2: [np.int64(17)]
Found overlapping subtitle for M20240925_1858_260319_0.fbx, segment 3: AARDBEI
Looking for handshape 'T' in matches for M20240925_1858_260319_0.fbx, segment 3
Correct handshapes for M20240925_1858_260319_0.fbx, segment 3: [np.int64(1)]
Found overlapping subtitle for M20240925_1858_260319

Evaluation

In [34]:
import numpy as np
import pandas as pd
from collections import Counter

# Options for multiple overlapping glosses:
#   "best_rank"   -> best (lowest) rank of all valid overlapping glosses
#   "max_overlap" -> only the gloss with the most overlap counts
#   "unique_only" -> segments with >1 overlapping gloss are discarded
POLICY = "best_rank"
MIN_OVERLAP_FRAMES = 3      # less overlap with a gloss doesn't count as overlap
KS = (1, 3, 5, 10, 20)
N_BOOT = 5000
LIBRARY_SIZE = None         # number of handshape classes in library; None = infer from matches_df
SEED = 42

In [ ]:
def _is_valid_handshape(hs):
    return hs is not None and not pd.isna(hs) and str(hs).strip() not in ("", "-")

# rank-lookup per (fbx, segment): {handshape_label: best rank}
rank_lookup = {}
for (fbx, seg), g in matches_df.groupby(["fbx_file", "segment"]):
    rank_lookup[(fbx, seg)] = g.groupby("handshape")["rank"].min().to_dict()


def overlap_frames(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start) + 1)


def collect_candidates(fbx_file, seg_idx):
    """All glosses that sufficiently overlap with the segment, including the rank of their handshape."""
    srt = load_srt(fbx_file)
    if srt is None:
        return None
    seg_start, seg_end = (int(x) for x in segments_by_fbx[fbx_file][seg_idx])
    ranks_here = rank_lookup.get((fbx_file, seg_idx), {})
    cands = []
    for sub in srt:
        s = get_frames_from_timecode(sub.start, fps)
        e = get_frames_from_timecode(sub.end, fps)
        ov = overlap_frames(seg_start, seg_end, s, e)
        if ov < MIN_OVERLAP_FRAMES:
            continue
        hs = get_handshape_from_gloss(sub.text)
        valid = _is_valid_handshape(hs)
        cands.append({
            "gloss": sub.text, "handshape": hs if valid else None, "valid": valid,
            "overlap": ov,
            # if not get, np.inf = handshape not in the returned list -> this counts as a miss
            "rank": ranks_here.get(hs, np.inf) if valid else np.nan,
        })
    return cands


def resolve(cands, policy):
    """Gives (chosen candidate, None) or (None, reason_for_exclusion)."""
    if cands is None:
        return None, "no_srt"
    if not cands:
        return None, "no_gloss_overlap"
    if policy == "unique_only":
        if len(cands) > 1:
            return None, "multiple_overlap"
        chosen = cands[0]
    elif policy == "max_overlap":
        chosen = max(cands, key=lambda c: c["overlap"])
    elif policy == "best_rank":
        valid = [c for c in cands if c["valid"]]
        if not valid:
            return None, "no_valid_handshape"
        chosen = min(valid, key=lambda c: c["rank"])
    else:
        raise ValueError(f"Onbekende policy: {policy}")
    if not chosen["valid"]:
        return None, "no_valid_handshape"
    return chosen, None


def build_eval_df(policy):
    rows, excluded = [], Counter()
    for fbx_file, segs in segments_by_fbx.items():
        for seg_idx in range(len(segs)):
            cands = collect_candidates(fbx_file, seg_idx)
            chosen, reason = resolve(cands, policy)
            if chosen is None:
                excluded[reason] += 1
                continue
            top1 = matches_df[(matches_df.fbx_file == fbx_file) &
                              (matches_df.segment == seg_idx) &
                              (matches_df["rank"] == 1)]["handshape"]
            rows.append({
                "fbx_file": fbx_file, "segment": seg_idx,
                "gt": chosen["handshape"], "rank": chosen["rank"],
                "pred_top1": top1.iloc[0] if len(top1) else None,
                "n_overlapping": len(cands),
            })
    return pd.DataFrame(rows), excluded

In [ ]:
def _per_row_stats(ranks):
    ranks = np.asarray(ranks, dtype=float)
    cols = [(ranks <= k).astype(float) for k in KS]
    cols.append(np.where(np.isinf(ranks), 0.0, 1.0 / ranks))   # reciprocal rank
    return np.column_stack(cols)                               # (n, len(KS)+1)

METRIC_NAMES = [f"top{k}" for k in KS] + ["MRR"]


def evaluate(df, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    per_clip_sum, per_clip_n = [], []
    for _, g in df.groupby("fbx_file"):
        per_clip_sum.append(_per_row_stats(g["rank"]).sum(axis=0))
        per_clip_n.append(len(g))
    S, N = np.array(per_clip_sum), np.array(per_clip_n, dtype=float)
    point = S.sum(axis=0) / N.sum()
    idx = rng.integers(0, len(N), size=(n_boot, len(N)))
    boot = S[idx].sum(axis=1) / N[idx].sum(axis=1, keepdims=True)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    out = pd.DataFrame({"estimate": point, "ci_low": lo, "ci_high": hi}, index=METRIC_NAMES)
    finite = df["rank"][np.isfinite(df["rank"])]
    extra = {
        "n_segments": len(df), "n_clips": len(N),
        "miss_rate (not in list)": float(np.isinf(df["rank"]).mean()),
        "median_rank (only from the found ones)": float(finite.median()) if len(finite) else np.nan,
    }
    return out, extra

In [ ]:
def baselines(df):
    L = LIBRARY_SIZE or matches_df["handshape"].nunique()
    res = {f"random top{k}": min(1.0, k / L) for k in KS}
    for k in KS:
        hits = []
        for clip in df["fbx_file"].unique():
            train = Counter(df[df.fbx_file != clip]["gt"])
            top = [h for h, _ in train.most_common(k)]
            hits += [g in top for g in df[df.fbx_file == clip]["gt"]]
        res[f"frequency top{k}"] = float(np.mean(hits))
    return pd.Series(res, name="baseline")

In [ ]:
eval_df, excluded = build_eval_df(POLICY)
metrics, extra = evaluate(eval_df)

print(f"=== Policy: {POLICY} ===")
print(metrics.round(3).to_string())
print("\nExtra info:", extra)
print("\nSkipped segments (reason -> count):", dict(excluded))
print("\nBaselines:")
print(baselines(eval_df).round(3).to_string())

=== Policy: best_rank ===
       estimate  ci_low  ci_high
top1      0.091   0.078    0.104
top3      0.208   0.190    0.227
top5      0.293   0.274    0.314
top10     0.433   0.412    0.455
top20     0.579   0.556    0.602
MRR       0.199   0.186    0.212

Overig: {'n_segments': 2042, 'n_clips': 398, 'miss_rate (niet in lijst)': 0.0881488736532811, 'median_rank (alleen gevonden)': 11.0}

Uitgesloten segmenten (reden -> aantal): {'no_valid_handshape': 244, 'no_gloss_overlap': 698}

Baselines:
random top1               0.021
random top3               0.064
random top5               0.106
random top10              0.213
random top20              0.426
frequentie-prior top1     0.336
frequentie-prior top3     0.609
frequentie-prior top5     0.720
frequentie-prior top10    0.859
frequentie-prior top20    0.956


In [ ]:
summary = []
for pol in ("best_rank", "max_overlap", "unique_only"):
    d, exc = build_eval_df(pol)
    m, ex = evaluate(d)
    row = {"policy": pol, "n_segments": ex["n_segments"], "excluded": sum(exc.values())}
    for name in METRIC_NAMES:
        row[name] = f"{m.loc[name,'estimate']:.2f} [{m.loc[name,'ci_low']:.2f}-{m.loc[name,'ci_high']:.2f}]"
    summary.append(row)
print(pd.DataFrame(summary).to_string(index=False))

# %% [7] Per klasse + top-1 confusion (voor foutanalyse)
per_class = (eval_df.assign(**{f"top{k}": eval_df["rank"] <= k for k in KS})
             .groupby("gt")[[f"top{k}" for k in KS]].mean()
             .join(eval_df.groupby("gt").size().rename("n")))
print(per_class.sort_values("n", ascending=False).round(2).to_string())

confusion = pd.crosstab(eval_df["gt"], eval_df["pred_top1"].fillna("?"),
                        rownames=["ground truth"], colnames=["top-1 prediction"])
print(confusion.to_string())

     policy  n_segments  excluded             top1             top3             top5            top10            top20              MRR
  best_rank        2042       942 0.09 [0.08-0.10] 0.21 [0.19-0.23] 0.29 [0.27-0.31] 0.43 [0.41-0.46] 0.58 [0.56-0.60] 0.20 [0.19-0.21]
max_overlap        1993       991 0.07 [0.06-0.08] 0.17 [0.15-0.19] 0.25 [0.23-0.26] 0.37 [0.35-0.40] 0.51 [0.49-0.54] 0.17 [0.16-0.18]
unique_only        1551      1433 0.06 [0.05-0.07] 0.16 [0.14-0.17] 0.24 [0.22-0.26] 0.36 [0.34-0.39] 0.52 [0.49-0.54] 0.16 [0.14-0.17]
                   top1  top3  top5  top10  top20    n
gt                                                    
1                  0.11  0.31  0.41   0.58   0.79  687
B                  0.02  0.15  0.28   0.45   0.55  321
5                  0.17  0.24  0.38   0.56   0.61  235
V                  0.06  0.13  0.18   0.32   0.50  114
S                  0.04  0.10  0.17   0.37   0.57  113
A                  0.03  0.05  0.05   0.13   0.30   76
T               